# 🧪 PubChem–ChEMBL Antimicrobial Assay Comparison

This notebook analyzes **antimicrobial bioassays** from PubChem and ChEMBL for a common set of pathogens of interest.

After building a pathogen–bioassay mapping pipeline, this notebook:

1. **Compares PubChem and ChEMBL assay coverage** per pathogen  
2. **Examines compound count consistency** between PubChem and ChEMBL  
3. **Classifies assays** by whether ChEMBL’s `cpds` matches PubChem’s:
- `Compounds_Tested`
- `Tested_Substances`
- neither  
4. **Assesses availability of active/inactive compound labels**

All analysis is done on **assays with matching ChEMBL IDs**, and aims to understand:
- How many assays are available per pathogen from both sources?
- How consistent are compound counts?
- Which assays have sufficient metadata for ML?

Final outputs are saved as `08_pubchem_vs_chembl_assays.csv`

## 0. Setup

In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import json
import requests
import re
from bs4 import BeautifulSoup
import seaborn as sns

In [2]:
# Project paths
NOTEBOOK_DIR = Path().resolve()

DATA_RAW = NOTEBOOK_DIR.parent / "data" / "raw"
DATA_PROCESSED = NOTEBOOK_DIR.parent / "data" / "processed"

PUBCHEM_DIR = DATA_RAW / "pubchem_bioassays"
DESC_DIR = PUBCHEM_DIR / "Description"
DATA_DIR = PUBCHEM_DIR / "Data"

KEEP_DIR = DATA_RAW / "filtered_assays"
KEEP_DESC = KEEP_DIR / "Description"
KEEP_DATA = KEEP_DIR / "Data"

In [3]:
pathogens = [
    "Acinetobacter baumannii", "Candida albicans", "Campylobacter",
    "Escherichia coli", "Enterococcus faecium", "Enterobacter",
    "Helicobacter pylori", "Klebsiella pneumoniae",
    "Mycobacterium tuberculosis", "Neisseria gonorrhoeae",
    "Pseudomonas aeruginosa", "Plasmodium falciparum",
    "Staphylococcus aureus", "Schistosoma mansoni",
    "Streptococcus pneumoniae"
]

## 04. PubChem-Chembl comparison

In [4]:
pubchem_assays = pd.read_csv(DATA_PROCESSED / "07_filtered_aids_metadata.csv")
pubchem_assays

,AID,Pathogen,PubChem_AID,ChEMBL_ID,Compounds_Tested,Compounds_Active,Compounds_Inactive,Tested_Substances,Target,Assay_Type,Assay_Format,Assay_Organism,Organism_TaxID,Assay_Strain,Organism_Target,Protein_Target,Source,pathogen_in_target,pathogen_in_organism,pathogen_match_any
0,1053491,Mycobacterium tuberculosis,1053491,CHEMBL3089115,1.0,NaN,NaN,1.0,Mycobacterium tuberculosis H37Rv,Binding,NaN,Mycobacterium tuberculosis,1773.0,NaN,NaN,Enoyl-[acyl-carrier-protein] reductase [NADH],ChEMBL,True,True,True
1,1053492,Mycobacterium tuberculosis,1053492,CHEMBL3089116,1.0,NaN,NaN,1.0,Mycobacterium tuberculosis H37Rv,Binding,NaN,Mycobacterium tuberculosis,1773.0,NaN,NaN,Enoyl-[acyl-carrier-protein] reductase [NADH],ChEMBL,True,True,True
2,1053493,Mycobacterium tuberculosis,1053493,CHEMBL3089485,1.0,NaN,NaN,1.0,Mycobacterium tuberculosis H37Rv,Binding,NaN,Mycobacterium tuberculosis,1773.0,NaN,NaN,Enoyl-[acyl-carrier-protein] reductase [NADH],ChEMBL,True,True,True
3,1053494,Mycobacterium tuberculosis,1053494,CHEMBL3089486,1.0,NaN,NaN,1.0,Mycobacterium tuberculosis H37Rv,Binding,NaN,Mycobacterium tuberculosis,1773.0,NaN,NaN,Enoyl-[acyl-carrier-protein] reductase [NADH],ChEMBL,True,True,True
4,1053495,Mycobacterium tuberculosis,1053495,CHEMBL3089487,1.0,NaN,NaN,1.0,Mycobacterium tuberculosis H37Rv,Binding,NaN,Mycobacterium tuberculosis,1773.0,NaN,NaN,Enoyl-[acyl-carrier-protein] reductase [NADH],ChEMBL,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133646,99225,Enterobacter,99225,CHEMBL706367,1.0,NaN,NaN,1.0,Enterobacter cloacae,Binding,NaN,Enterobacter cloacae,550.0,NaN,NaN,Beta-lactamase,ChEMBL,True,True,True
133647,99226,Enterobacter,99226,CHEMBL706368,1.0,NaN,NaN,1.0,Enterobacter cloacae,Binding,NaN,Enterobacter cloacae,550.0,NaN,NaN,Beta-lactamase,ChEMBL,True,True,True
133648,99234,Plasmodium falciparum,99234,CHEMBL706376,10.0,8.0,NaN,10.0,Homo sapiens (human),Binding,NaN,Plasmodium falciparum,5833.0,NaN,NaN,L-lactate dehydrogenase A chain,ChEMBL,False,True,True
133649,99235,Plasmodium falciparum,99235,CHEMBL880830,4.0,4.0,NaN,4.0,NaN,Binding,NaN,Plasmodium falciparum,5833.0,NaN,NaN,NaN,ChEMBL,False,True,True


In [5]:
chembl_assays = pd.read_csv(DATA_PROCESSED / "ChEMBL_bioassays/assays_ChEMBL_all.csv") # This file was produced by Arnau
chembl_assays

,assay_id,assay_type,assay_organism,doc_chembl_id,target_type,target_chembl_id,target_organism,activity_type,unit,activities,nan_values,cpds,pathogen
0,CHEMBL4296188,F,Acinetobacter baumannii,CHEMBL4296181,ORGANISM,CHEMBL614425,Acinetobacter baumannii,INHIBITION,%,21582,26,21532,abaumannii
1,CHEMBL4296193,F,Acinetobacter baumannii,CHEMBL4296181,ORGANISM,CHEMBL614425,Acinetobacter baumannii,INHIBITION,%,1566,0,1565,abaumannii
2,CHEMBL4296188,F,Acinetobacter baumannii,CHEMBL4296181,ORGANISM,CHEMBL614425,Acinetobacter baumannii,MIC,umol.L-1,1062,0,1041,abaumannii
3,CHEMBL1640427,F,Acinetobacter baumannii,CHEMBL1629439,ORGANISM,CHEMBL614425,Acinetobacter baumannii,MIC,umol.L-1,141,0,140,abaumannii
4,CHEMBL2329975,F,Acinetobacter baumannii,CHEMBL2321876,ORGANISM,CHEMBL614425,Acinetobacter baumannii,ACTIVITY,NaN,86,86,86,abaumannii
...,...,...,...,...,...,...,...,...,...,...,...,...,...
145594,CHEMBL1676627,F,Streptococcus pneumoniae,CHEMBL1671689,ORGANISM,CHEMBL347,Streptococcus pneumoniae,MIC50,umol.L-1,1,0,1,spneumoniae
145595,CHEMBL1676627,F,Streptococcus pneumoniae,CHEMBL1671689,ORGANISM,CHEMBL347,Streptococcus pneumoniae,MIC,umol.L-1,1,0,1,spneumoniae
145596,CHEMBL1676627,F,Streptococcus pneumoniae,CHEMBL1671689,ORGANISM,CHEMBL347,Streptococcus pneumoniae,MIC90,umol.L-1,1,0,1,spneumoniae
145597,CHEMBL1676628,F,Streptococcus pneumoniae,CHEMBL1671689,ORGANISM,CHEMBL347,Streptococcus pneumoniae,MIC50,umol.L-1,1,0,1,spneumoniae


In [ ]:
chembl_pathogens = sorted(chembl_assays["pathogen"].dropna().unique())
chembl_pathogens

In [8]:
# Define mapping from ChEMBL codes to canonical names
pathogen_map = {
    "abaumannii": "Acinetobacter baumannii",
    "calbicans": "Candida albicans",
    "ecoli": "Escherichia coli",
    "efaecium": "Enterococcus faecium",
    "enterobacter": "Enterobacter",
    "hpylori": "Helicobacter pylori",
    "kpneumoniae": "Klebsiella pneumoniae",
    "mtuberculosis": "Mycobacterium tuberculosis",
    "ngonorrhoeae": "Neisseria gonorrhoeae",
    "paeruginosa": "Pseudomonas aeruginosa",
    "pfalciparum": "Plasmodium falciparum",
    "saureus": "Staphylococcus aureus",
    "smansoni": "Schistosoma mansoni",
    "spneumoniae": "Streptococcus pneumoniae",
}

In [9]:
# Map ChEMBL shorthand to canonical names
chembl_assays["Pathogen"] = chembl_assays["pathogen"].map(pathogen_map)

# Drop rows where the mapping failed (optional)
chembl_assays = chembl_assays.dropna(subset=["Pathogen"])

### 4.1. Bioassay counts

In [10]:
# PubChem
pubchem_counts = (
    pubchem_assays
    .groupby("Pathogen")["AID"]
    .nunique()
    .reset_index(name="PubChem_Assays")
)

# ChEMBL (now with Pathogen column)
chembl_counts = (
    chembl_assays
    .groupby("Pathogen")["assay_id"]
    .nunique()
    .reset_index(name="ChEMBL_Assays")
)

# Merge + format
summary = (
    pubchem_counts
    .merge(chembl_counts, on="Pathogen", how="outer")
    .fillna(0)
)

summary[["PubChem_Assays", "ChEMBL_Assays"]] = summary[["PubChem_Assays", "ChEMBL_Assays"]].astype(int)
summary["Total"] = summary["PubChem_Assays"] + summary["ChEMBL_Assays"]
summary = summary.sort_values("Total", ascending=False).reset_index(drop=True)

In [ ]:
# Data
labels = summary["Pathogen"].values
pubchem_vals = summary["PubChem_Assays"].values
chembl_vals = summary["ChEMBL_Assays"].values

N = len(labels)
x = np.arange(N)

# ⏩ Bar styling
bar_width = 0.35
bar_spacing = 0.2

color_pubchem = "#AA96FA"  # purple 
color_chembl  = "#BEE6B4"    # Green

# Plot
plt.figure(figsize=(14, 6))

plt.bar(x - bar_spacing, pubchem_vals, width=bar_width, color=color_pubchem,
        ec="black", zorder=2, label="PubChem Assays")

plt.bar(x + bar_spacing, chembl_vals, width=bar_width, color=color_chembl,
        ec="black", zorder=2, label="ChEMBL Assays")

# Add labels
for i in range(N):
    plt.text(x[i] - bar_spacing, pubchem_vals[i] + 1, str(pubchem_vals[i]),
             ha='center', va='bottom', fontsize=8)
    plt.text(x[i] + bar_spacing, chembl_vals[i] + 1, str(chembl_vals[i]),
             ha='center', va='bottom', fontsize=8)

# Aesthetics
plt.xticks(x, labels, rotation=45, ha="right")
plt.ylabel("Number of Assays")
plt.title("Number of Assays per Pathogen (PubChem vs ChEMBL)")
plt.grid(axis="y", linestyle="--", zorder=1)
plt.ylim(0, max(pubchem_vals.max(), chembl_vals.max()) * 1.2)

plt.legend(loc="upper right", framealpha=1, edgecolor="k", prop={"size": 10})
plt.tight_layout()
plt.show()

### 4.1. % Chembl_ID present in PubChem bioassays

In [ ]:
# 1. Filter rows with ChEMBL IDs
aids_with_chembl = pubchem_assays[pubchem_assays["ChEMBL_ID"].notna()]

# 2. Group total AIDs and AIDs with ChEMBL by pathogen
total_aids = pubchem_assays.groupby("Pathogen")["AID"].nunique().reset_index(name="Total_AIDs")
with_chembl = aids_with_chembl.groupby("Pathogen")["AID"].nunique().reset_index(name="AIDs_with_ChEMBL")

# 3. Merge and compute percentage
summary = total_aids.merge(with_chembl, on="Pathogen", how="left")
summary["AIDs_with_ChEMBL"] = summary["AIDs_with_ChEMBL"].fillna(0).astype(int)
summary["Percent_ChEMBL_IDs"] = 100 * summary["AIDs_with_ChEMBL"] / summary["Total_AIDs"]
summary = summary.sort_values("Percent_ChEMBL_IDs", ascending=False).reset_index(drop=True)

In [ ]:
# Data
labels = summary["Pathogen"].values
percent_chembl = summary["Percent_ChEMBL_IDs"].values

N = len(labels)
x = np.arange(N)

# Plot settings
bar_width = 0.6
color_chembl = "#AA96FA"

plt.figure(figsize=(14, 6))

bars = plt.bar(x, percent_chembl, width=bar_width,
               color=color_chembl, ec="black", zorder=2,
               label="% AIDs with ChEMBL ID")

# Add labels on top
for i in range(N):
    plt.text(x[i], percent_chembl[i] + 2, f"{percent_chembl[i]:.1f}%",
             ha='center', va='bottom', fontsize=9, zorder=3)

# Aesthetics
plt.xticks(x, labels, rotation=45, ha="right")
plt.ylabel("Percentage (%)")
plt.ylim(0, 110)
plt.title("Percentage of AIDs with ChEMBL ID per Pathogen")
plt.grid(axis="y", linestyle="--", zorder=1)
plt.tight_layout()
plt.show()

### 4.2. Number of compounds per assay

In PubChem terminology, a **substance** is a chemical sample description provided by a single source and a **compound** is a normalized chemical structure representation found in one or more contributed substances.

PubChem calls the community-provided sample descriptions *substances*.  Each record found in the PubChem **Substance database** (https://www.ncbi.nlm.nih.gov/pcsubstance) contains information provided by an individual contributor about a particular chemical substance.  Substance records are independent of each other.  Two different Substance records (from the same or different providers) could provide different information about the same chemical structure.  

For example, one substance record may give information about the biological role of aspirin, while another may give information about a research grade sample of aspirin.  The Substance database maintains the provenance of chemical substance information in PubChem.  It helps users see who provided what.  

As a result, there may be many substance records about a given molecule, presenting a problem for users who are interested in an aggregated view of information on the molecule.  This is where the PubChem Compound database (https://www.ncbi.nlm.nih.gov/pccompound) comes into play.

The **Compound database** is derived from the chemical structure contents found in the Substance database.  Each chemical is computationally examined with a series of validation and normalization steps.  This process results in a normalized representation of the chemical structure for a substance record.  

Chemical substances in the Substance database that are not completely described or that fail normalization procedures are not included in the Compound database.  Those substances in the Substance database that pass chemical structure normalization procedures are linked to a “compound” record in the Compound database.  If two substances refer to the same chemical structure, they point to the same compound.  This allows data from different Substance data providers to be aggregated through a common Compound record.  However, also having separate substance records is still valuable to users, who, for example, might be interested in the provenance of a substance or a particular state of the chemical (e.g., a different tautomeric form).  

In essence, a primary purpose of the PubChem Compound database is to provide a “non-redundant” view of the depositor-contributed chemical structure contents stored in the PubChem Substance database.

In [ ]:
pubchem_assays = pd.read_csv(DATA_PROCESSED / "07_filtered_aids_metadata.csv")
chembl_assays = pd.read_csv(DATA_PROCESSED / "ChEMBL_bioassays/assays_ChEMBL_all.csv") 

In [ ]:
# From PubChem
pubchem_sub = pubchem_assays[
    [
        "Pathogen",
        "PubChem_AID",
        "ChEMBL_ID",
        "Compounds_Tested",
        "Compounds_Active",
        "Compounds_Inactive",
        "Tested_Substances"
    ]
]

# Keep only assays with a ChEMBL_ID in PubChem
pubchem_sub = pubchem_sub[
    pubchem_sub["ChEMBL_ID"].notna()
]

# Keep only assays with a compound counts available
pubchem_sub = pubchem_sub[
    pubchem_sub["Compounds_Tested"].notna()
]

# From ChEMBL
chembl_sub = chembl_assays[
    ["assay_id", "cpds"]
]

# Rename assay_id → ChEMBL_ID for merging
chembl_sub = chembl_sub.rename(columns={"assay_id": "ChEMBL_ID"})

In [ ]:
# Merge on ChEMBL_ID (We exclude assays only present either in PubChem or in Chembl selection)

compounds = (
    pubchem_sub
    .merge(
        chembl_sub,
        on="ChEMBL_ID",
        how="inner"
    )
)

In [ ]:
# Ensure numeric types

num_cols = ["Compounds_Tested", "Tested_Substances", "cpds"]

for col in num_cols:
    compounds[col] = pd.to_numeric(compounds[col], errors="coerce")

# Compute compound difference
compounds["Compounds_diff"] = (
    compounds["Compounds_Tested"] - compounds["cpds"]
)

compounds["Substances_diff"] = (
    compounds["Tested_Substances"] - compounds["cpds"]
)

compounds

In [ ]:
# How many assays have different compound counts
assays_compunds_diff = compounds[compounds["Compounds_diff"] != 0]
assays_substances_diff = compounds[compounds["Substances_diff"] != 0]

print(len(assays_compunds_diff))
print(len(assays_substances_diff))

In [ ]:
assays_compunds_diff.groupby("Pathogen").size().sort_values(ascending=False)

In [ ]:
assays_substances_diff.groupby("Pathogen").size().sort_values(ascending=False)

In [ ]:
# Classification of bioassays depending on whether cpds matches either Compounds_Tested or Tested_Substances.

def classify_assay(row):
    cpds = row["cpds"]
    comp = row["Compounds_Tested"]
    subs = row["Tested_Substances"]

    if pd.isna(cpds):
        return "no_cpds"

    matches_comp = not pd.isna(comp) and cpds == comp
    matches_subs = not pd.isna(subs) and cpds == subs

    if matches_comp and matches_subs:
        return "cpds_match_both"
    elif matches_comp:
        return "cpds_match_only_compounds"
    elif matches_subs:
        return "cpds_match_only_substances"
    else:
        return "cpds_mismatch"

compounds["cpds_match_class"] = compounds.apply(classify_assay, axis=1)

In [ ]:
compounds["cpds_match_class"].value_counts()

In [ ]:
# Count assays per pathogen and class
counts = compounds.groupby(["Pathogen", "cpds_match_class"]).size().unstack(fill_value=0)

# Define order and colors
match_order = [
    "cpds_match_both",
    "cpds_match_only_compounds",
    "cpds_match_only_substances",
    "cpds_mismatch",
]
colors = {
    "cpds_match_both": "#BEE6B4",             # Green
    "cpds_match_only_compounds": "#AA96FA",   # Purple
    "cpds_match_only_substances": "#FAD782",  # Yellow
    "cpds_mismatch": "#FAA08B",               # Orange
}

# Reorder columns and normalize to percentage
counts = counts.reindex(columns=match_order, fill_value=0)
percentages = counts.div(counts.sum(axis=1), axis=0) * 100

# Plot
labels = percentages.index.tolist()
N = len(labels)
x = np.arange(N)
bar_width = 0.8

plt.figure(figsize=(14, 6))

bottom = np.zeros(N)

for match_class in match_order:
    plt.bar(
        x,
        percentages[match_class],
        bottom=bottom,
        width=bar_width,
        color=colors[match_class],
        ec="black",
        label=match_class.replace("_", " ").title(),
        zorder=2
    )
    bottom += percentages[match_class].values

# Aesthetics
plt.xticks(x, labels, rotation=45, ha="right")
plt.ylabel("Percentage of Assays (%)")
plt.title("Pubchem (compound/substance) vs Chembl (cpds) counts match")
plt.ylim(0, 100)
plt.grid(axis="y", linestyle="--", zorder=1)
plt.legend(loc="lower left", framealpha=1, edgecolor="k", prop={"size": 9})
plt.tight_layout()
plt.show()

In [ ]:
# Filter for mismatch cases
mismatch = compounds[
    (compounds["cpds_match_class"] == "cpds_mismatch") &
    compounds["Compounds_diff"].notna()
]

# Plot
plt.figure(figsize=(14, 6))

sns.stripplot(
    data=mismatch,
    x="Pathogen",
    y="Compounds_diff",
    jitter=True,
    alpha=0.7,
    size=6,
    linewidth=0.5,
    edgecolor="black",
    color="#FAA08B"  # Match the orange used for mismatch
)

plt.axhline(0, linestyle="--", color="gray")
plt.title("Compound Count Difference per Assay (Only Mismatches)", fontsize=14)
plt.ylabel("Compound Count Difference (PubChem − ChEMBL)", fontsize=12)
plt.xlabel("Pathogen", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Filter and sort
mismatch_sorted = compounds[
    (compounds["cpds_match_class"] == "cpds_mismatch") &
    (compounds["Compounds_diff"].notna())
].sort_values("Compounds_diff", ascending=False)

# Show the full DataFrame with all columns
mismatch_sorted.reset_index(drop=True, inplace=True)
mismatch_sorted

### 4.3. Active vs Inactive Compounds

In [ ]:
# Classify each assay
def classify_activity(row):
    active = pd.notna(row["Compounds_Active"])
    inactive = pd.notna(row["Compounds_Inactive"])

    if active and inactive:
        return "both"
    elif active:
        return "only_active"
    elif inactive:
        return "only_inactive"
    else:
        return "none"

# Apply classification
compounds["activity_info_class"] = compounds.apply(classify_activity, axis=1)

# Count per pathogen and class
counts = compounds.groupby(["Pathogen", "activity_info_class"]).size().unstack(fill_value=0)

# Define class order and colors
class_order = ["both", "only_active", "only_inactive", "none"]
colors = {
    "both": "#BEE6B4",         # Green
    "only_active": "#AA96FA",  # Purple
    "only_inactive": "#FAD782",# Yellow
    "none": "#FAA08B",         # Red-ish
}

# Reindex and convert to percentages
counts = counts.reindex(columns=class_order, fill_value=0)
percentages = counts.div(counts.sum(axis=1), axis=0) * 100

# Plotting
labels = percentages.index.tolist()
N = len(labels)
x = np.arange(N)
bar_width = 0.8

plt.figure(figsize=(14, 6))
bottom = np.zeros(N)

for cls in class_order:
    plt.bar(
        x,
        percentages[cls],
        bottom=bottom,
        width=bar_width,
        color=colors[cls],
        edgecolor="black",
        label=cls.replace("_", " ").title(),
        zorder=2
    )
    bottom += percentages[cls].values

# Aesthetics
plt.xticks(x, labels, rotation=45, ha="right")
plt.ylabel("Percentage of Assays (%)")
plt.title("Presence of Active/Inactive Compound Info in Pubchem")
plt.ylim(0, 100)
plt.grid(axis="y", linestyle="--", zorder=1)
plt.legend(loc="upper left", framealpha=1, edgecolor="k", prop={"size": 9})
plt.tight_layout()
plt.show()

In [ ]:
compounds.to_csv(DATA_PROCESSED / "08_pubchem_vs_chembl_assays.csv", index=False)